# 2. Preprocesamiento

Este notebook lee la tabla Delta creada en el notebook anterior, aplica transformaciones requeridas para entrenamiento y registra las variables generadas como features en un contexto de Feature Store.

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

spark = (
    SparkSession.builder
    .appName("control-2-preprocesamiento")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

bronze_df = spark.table("bronze_control_2")
bronze_df.printSchema()
bronze_df.show(5, truncate=False)

In [ ]:
transformed_df = (
    bronze_df
    .withColumn("feature_1_norm", F.col("feature_1") / F.lit(10.0))
    .withColumn("feature_2_norm", F.col("feature_2") / F.lit(10.0))
    .withColumn("feature_3_norm", F.col("feature_3") / F.lit(20.0))
    .withColumn("feature_sum", F.col("feature_1_norm") + F.col("feature_2_norm") + F.col("feature_3_norm"))
    .withColumn("target", F.col("target").cast("int"))
    .withColumn("processed_at", F.current_timestamp())
)

transformed_df.printSchema()
transformed_df.show(5, truncate=False)

transformed_df.write.mode("overwrite").format("delta").saveAsTable("silver_control_2")
print("Datos procesados guardados en Delta como silver_control_2")

feature_df = transformed_df.select(
    "feature_1_norm",
    "feature_2_norm",
    "feature_3_norm",
    "feature_sum",
    "target"
)

try:
    from databricks.feature_store import FeatureStoreClient

    fs = FeatureStoreClient()
    fs.create_table(
        name="control_2_feature_store",
        primary_keys=["target"],
        schema=feature_df.schema,
        description="Features generadas para el proyecto control-2",
        df=feature_df
    )
    print("Feature Store: tabla control_2_feature_store creada correctamente.")
except Exception as e:
    print(f"Feature Store no disponible en este entorno: {e}")
    feature_df.write.mode("overwrite").format("delta").saveAsTable("features_control_2")
    print("Se guardó una copia de features en Delta como features_control_2")